In [9]:
from PIL import Image, ImageDraw, ImageFont

In [1]:
# import sys
# from pathlib import Path
# import numpy as np

# REPO_ROOT = Path.cwd().parent.parent          # examples/my_video_jepa -> repo root

# # the eb_jepa/datasets/ package holds moving_mnist.py (the copy we want to import)
# sys.path.insert(0, str(REPO_ROOT / "eb_jepa" / "datasets"))

# # the actual .npy data lives in the repo-root datasets/ folder
# DATA_FILE = REPO_ROOT / "datasets" / "mnist_test_seq.npy"

# import numpy as np 

# dataset = np.load(DATA_FILE)

# dataset = np.swapaxes(dataset,0,1) # Converting (T,N, H,W) --> (N,T,H,W)
# train_data =dataset[:9000]
# train_data = np.reshape(
#     train_data, [train_data.shape[0] * 2, train_data.shape[1]//2, train_data.shape[2], train_data.shape[3]]
# )
# print(train_data.shape) # (N, T, H, W)

In [14]:
from eb_jepa.datasets.moving_mnist import MovingMNISTDet
from torch.utils.data import DataLoader

train_data = MovingMNISTDet(split="train")
train_dl = DataLoader(train_data, batch_size=32, shuffle=False,drop_last=True)

In [17]:
for batch in train_dl: 
    x = batch["video"]
    print(x.shape)
    break

torch.Size([32, 1, 10, 64, 64])


In [ ]:
import torch 
from architectures import ResNet5

D_OBS = 1  # input channels (gray scale) 
H_ENC = 32 # hidden dim in encoder 
D_STC = 16 # represenation dim (encoder output channels) 
H_PRE = 32   # hidden dim in predictor

encoder = ResNet5(in_d=D_OBS, h_d=H_ENC, out_d= D_STC)
with torch.no_grad(): 
    x_jepa  = encoder(x)
    print(x_jepa.shape)

torch.Size([32, 16, 10, 64, 64])


In [20]:
T = x.shape[2]
print(T)

10


In [19]:
from architectures import ResUNet, StateOnlyPredictor, Projector
from losses import VCLoss, SquareLossSeq
from jepa import JEPA 

# --- Predictor: [prev, next] representations → predicted next ---
predictor_net = ResUNet(2 * D_STC, H_PRE, D_STC)        # input is 2x D_STC (two frames concatenated)
predictor     = StateOnlyPredictor(predictor_net, context_length=2)

# --- Projector: used inside the losses (not on the encoder output) ---
projector = Projector(f"{D_STC}-{D_STC*4}-{D_STC*4}")   # "16-64-64"

# --- Losses ---
# They both have optional projector 
regularizer = VCLoss(std_coeff=10.0, cov_coeff=100.0, proj=projector)  # prevents collapse
pred_loss   = SquareLossSeq(projector)                                  # MSE in projected space

# --- JEPA: ties everything together ---
jepa = JEPA(encoder, encoder, predictor, regularizer, pred_loss)

In [ ]:
preds,_ = jepa.unroll(
    x, 
    actions = None,
    nsteps = T - 2, 
    unroll_mode= "parallel",
    compute_loss=False,
    return_all_steps=True
)